[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HyeonjeYang/einthovens_triangle/blob/main/einthoven_triangle.ipynb)

In [ ]:
# --- Colab compatibility (safe no-op on regular Jupyter / VS Code / etc.) ---
# Colab already ships numpy/matplotlib/tqdm/ipywidgets, but its interactive-widget
# frontend needs to be explicitly enabled for sliders/Play to render reliably.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")

# Einthoven's Triangle — Interactive ECG Vector Simulator

The heart is modeled as a single dipole at the center of a homogeneous, spherical torso.
Under this assumption, the potential at each limb electrode is proportional to the
projection of the cardiac vector onto the direction from the center to that electrode —
this is exactly Einthoven's assumption, and it's why a lead voltage is just a cosine projection.

- **Adjust:** heart rate (HR), cardiac condition, and simulated recording-equipment settings
  (amplifier **gain** and **noise** level).
- **See:** the instantaneous heart vector inside the triangle, its projection onto leads I/II/III,
  and the resulting lead waveforms.
- **Play/Time** slider animates one cardiac cycle; pause and drag manually for a static snapshot.

*For education only — not a diagnostic tool.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout
from IPython.display import display
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

### Geometry & physics assumptions

1. Heart = single fixed dipole at the sphere's center.
2. Torso = homogeneous, spherical volume conductor.
3. RA / LA / LL electrodes are equidistant from the center → equilateral triangle.
4. ⇒ **Lead voltage = |heart vector| · cos(angle between vector and lead axis)**.

Lead axes below are *derived* from the electrode positions (not hard-coded), so the
usual textbook angles (I = 0°, II = 60°, III = 120°) fall out automatically.

In [ ]:
# --- Einthoven's triangle geometry (unit circle = torso cross-section) ---
_H = np.sqrt(3) / 2            # exact equilateral-triangle constant (avoids rounding drift)
RA = np.array([-_H, 0.5])      # right arm
LA = np.array([ _H, 0.5])      # left arm
LL = np.array([0.0, -1.0])     # left leg

def unit(v):
    return v / np.linalg.norm(v)

# Lead vector points from the (-) electrode to the (+) electrode
LEAD_VECS = {
    "I":   unit(LA - RA),
    "II":  unit(LL - RA),
    "III": unit(LL - LA),
}

def clinical_to_xy(angle_deg, magnitude=1.0):
    """Clinical frontal-plane axis (0deg = Lead I, +90deg = inferior/aVF, clockwise-positive)
    -> (x, y) in the plotting plane (y-axis up)."""
    theta = np.deg2rad(-angle_deg)
    return magnitude * np.array([np.cos(theta), np.sin(theta)])

### Cardiac condition presets

Each condition sets a representative heart rate, mean QRS axis, and waveform
morphology (irregular rhythm, absent P wave, inverted T, pathologic Q). This is a
teaching simplification, not a clinically accurate model of any single diagnosis.

In [ ]:
CONDITIONS = {
    "Normal Sinus Rhythm":   dict(hr=75,  axis=60,  amp=1.0, irregular=False, has_p=True,  t_pol= 1, q_boost=1.0),
    "Sinus Tachycardia":     dict(hr=130, axis=60,  amp=1.0, irregular=False, has_p=True,  t_pol= 1, q_boost=1.0),
    "Sinus Bradycardia":     dict(hr=45,  axis=60,  amp=1.0, irregular=False, has_p=True,  t_pol= 1, q_boost=1.0),
    "Atrial Fibrillation":   dict(hr=100, axis=60,  amp=0.9, irregular=True,  has_p=False, t_pol= 1, q_boost=1.0),
    "Myocardial Infarction": dict(hr=80,  axis=-40, amp=0.6, irregular=False, has_p=True,  t_pol=-1, q_boost=2.5),
    "Left Axis Deviation":   dict(hr=75,  axis=-45, amp=1.2, irregular=False, has_p=True,  t_pol= 1, q_boost=1.0),
    "Right Axis Deviation":  dict(hr=75,  axis=120, amp=1.0, irregular=False, has_p=True,  t_pol= 1, q_boost=1.0),
}

### Waveform model

The heart vector at any instant is the sum of a few directional Gaussian "bumps"
(P, Q, R, S, T), each with its own timing, width, amplitude, and axis. Summing
projections is equivalent to projecting the sum (linearity), so the triangle view and
the lead traces are always mutually consistent — including Einthoven's law (II = I + III).

In [ ]:
def _fibrillation_noise(seed=7, n=5):
    """Fixed, irregular fibrillatory baseline used instead of an organized P wave (AFib)."""
    rng = np.random.default_rng(seed)
    freqs = rng.uniform(8, 20, n)
    phases = rng.uniform(0, 2 * np.pi, n)
    axes = rng.uniform(0, 360, n)
    amps = rng.uniform(0.02, 0.06, n)

    def f(t_ms):
        v = np.zeros(2)
        for fr, ph, ax, am in zip(freqs, phases, axes, amps):
            v += clinical_to_xy(ax, am * np.sin(2 * np.pi * fr * (t_ms / 1000.0) + ph))
        return v
    return f

FIB_NOISE = _fibrillation_noise()

def vector_at_phase(t_ms, rr_ms, cond, amp_scale):
    """Instantaneous heart vector at time t_ms since this beat started."""
    total = np.zeros(2)

    def bump(t0_frac, w_frac, amp, axis_offset=0.0):
        t0, w = t0_frac * rr_ms, max(w_frac * rr_ms, 1e-6)
        g = np.exp(-0.5 * ((t_ms - t0) / w) ** 2)
        return clinical_to_xy(cond["axis"] + axis_offset, g * amp * amp_scale)

    if cond["has_p"]:
        total += bump(0.15, 0.030, 0.15)
    else:
        total += FIB_NOISE(t_ms) * amp_scale
    total += bump(0.28, 0.012, -0.10 * cond["q_boost"], axis_offset=-30)   # Q
    total += bump(0.30, 0.018,  1.00 * cond["amp"],                     )  # R (main)
    total += bump(0.33, 0.014, -0.20,                  axis_offset= 45)   # S
    t_axis_off = 0.0 if cond["t_pol"] > 0 else 180.0
    total += bump(0.55, 0.060,  0.30 * cond["t_pol"] * cond["amp"], axis_offset=t_axis_off)  # T
    return total

def beat_times(hr, irregular, n_beats=3, seed=42):
    """Start times + individual RR interval (ms) for each beat in the strip."""
    base_rr = 60000.0 / hr
    if not irregular:
        rr_arr = np.full(n_beats, base_rr)
    else:
        rng = np.random.default_rng(seed)
        rr_arr = base_rr * rng.uniform(0.6, 1.4, n_beats)   # "irregularly irregular"
    starts = np.concatenate([[0.0], np.cumsum(rr_arr)[:-1]])
    return starts, rr_arr

def vector_at_time(t_abs_arr, starts, rr_arr, cond, amp_scale):
    """Vectorized heart vector for an array of absolute times (ms)."""
    t_abs_arr = np.atleast_1d(t_abs_arr)
    idx = np.clip(np.searchsorted(starts, t_abs_arr, side="right") - 1, 0, len(starts) - 1)
    out = np.zeros((len(t_abs_arr), 2))
    for i, (t_abs, ix) in enumerate(zip(t_abs_arr, idx)):
        out[i] = vector_at_phase(t_abs - starts[ix], rr_arr[ix], cond, amp_scale)
    return out

### Simulated recording equipment: gain & noise

Real ECG amplifiers have a **gain** setting (signal amplification) and pick up **noise**
(EMG/muscle artifact, baseline wander, mains interference) that a filter only partly removes.
These are equipment properties, not properties of the heart itself, so they are modeled
separately from the cardiac vector above: gain simply scales the vector, and noise is added
independently to each lead trace afterward (real electrode noise is not shared across leads).

In [ ]:
def lead_noise(t_arr, level, lead_idx):
    """Simulated per-lead recording noise: EMG-like + baseline wander + 60 Hz mains."""
    if level <= 0:
        return np.zeros_like(t_arr)
    rng = np.random.default_rng(900 + lead_idx)   # fixed seed -> stable, reproducible trace
    hf = rng.standard_normal(len(t_arr))
    wander = np.sin(2 * np.pi * 0.25 * t_arr / 1000.0 + lead_idx)
    mains = np.sin(2 * np.pi * 60 * t_arr / 1000.0)
    return level * (0.6 * hf + 0.5 * wander + 0.3 * mains)

In [ ]:
def compute_leads(hr, gain, noise, cond, n_points=600):
    """Shared computation: beat timing + per-lead voltage traces (gain applied, noise added).
    Used by both the triangle/lead-panel view and the classic ECG-strip view below, so the
    two stay numerically consistent by construction."""
    amp_scale = float(gain)
    starts, rr_arr = beat_times(hr, cond["irregular"])
    window_ms = starts[-1] + rr_arr[-1]
    t_arr = np.linspace(0, window_ms, n_points)
    vecs = vector_at_time(t_arr, starts, rr_arr, cond, amp_scale)
    leads = {k: vecs @ LEAD_VECS[k] for k in ("I", "II", "III")}
    for i, k in enumerate(("I", "II", "III")):
        leads[k] = leads[k] + lead_noise(t_arr, noise, i)
    return t_arr, leads, starts, rr_arr, window_ms

In [ ]:
# Sanity check: Einthoven's law (II = I + III) must hold for every condition/time point
for name, cond in tqdm(CONDITIONS.items(), desc="Validating conditions"):
    starts, rr_arr = beat_times(cond["hr"], cond["irregular"])
    t_arr = np.linspace(0, starts[-1] + rr_arr[-1], 200)
    vecs = vector_at_time(t_arr, starts, rr_arr, cond, amp_scale=1.0)
    I, II, III = (vecs @ LEAD_VECS[k] for k in ("I", "II", "III"))
    assert np.allclose(II, I + III, atol=1e-8), f"Einthoven's law violated for {name}"
print("OK: Einthoven's law (II = I + III) holds for all conditions.")

### Validity notes (kept intentionally simple — this is a teaching tool)

- **Sphere + dipole assumption**: the classical simplification behind Einthoven's triangle
  itself, not real torso anatomy — this is what the whole model is built on, so it's exact
  *within* this framework.
- **HR / axis presets**: chosen to sit within typical textbook ranges (e.g. normal axis ~59°,
  LAD < -30°, RAD > +90°, sinus tach > 100 bpm, sinus brady < 60 bpm, AFib = irregularly
  irregular with no discrete P wave). They represent one illustrative example per condition,
  not a diagnostic rule.
- **Gain and noise are equipment settings, not physiology** — they intentionally do *not*
  scale with any physiological parameter (earlier versions linked amplitude to blood
  pressure, which isn't a real electrical relationship, so that link was removed).
- With noise > 0, Lead II may no longer equal Lead I + Lead III exactly, because real
  electrode/amplifier noise is independent per lead — the noise-free vector above always
  satisfies Einthoven's law exactly.

### Controls

- **Condition** — preset rhythm / axis / morphology (also sets a representative HR)
- **HR** — freely adjustable
- **Gain / Noise** — simulated amplifier gain and recording-noise level (equipment, not physiology)
- **Play / Time** — scrub or animate through the current beat window
- **Peak buttons** — jump to the P, QRS, or T peak of the first beat

In [ ]:
hr_slider = widgets.IntSlider(value=75, min=30, max=220, description="HR (bpm)",
                               continuous_update=False, style={"description_width": "initial"})
gain_slider = widgets.FloatSlider(value=1.0, min=0.3, max=2.5, step=0.1, description="Amp Gain (x)",
                                   continuous_update=False, style={"description_width": "initial"})
noise_slider = widgets.FloatSlider(value=0.0, min=0.0, max=0.3, step=0.01, description="Noise level",
                                    continuous_update=False, style={"description_width": "initial"})
cond_dd = widgets.Dropdown(options=list(CONDITIONS.keys()), value="Normal Sinus Rhythm", description="Condition")

time_slider = widgets.FloatSlider(value=0, min=0, max=1, step=1, description="Time (ms)",
                                   continuous_update=True, layout=Layout(width="480px"))
play = widgets.Play(value=0, min=0, max=1, step=10, interval=150)
widgets.link((play, "value"), (time_slider, "value"))  # plain link: syncs through the kernel

btn_p = widgets.Button(description="P peak", layout=Layout(width="90px"))
btn_qrs = widgets.Button(description="QRS peak", layout=Layout(width="90px"))
btn_t = widgets.Button(description="T peak", layout=Layout(width="90px"))

In [ ]:
out = widgets.Output()

# Build the figure once and reuse it (clear + redraw) — much faster than creating a
# new Figure every call, which matters once Play is animating many frames per second.
_fig = plt.figure(figsize=(11, 5.5))
_gs = _fig.add_gridspec(3, 2, width_ratios=[1, 1.3])
_ax_tri = _fig.add_subplot(_gs[:, 0])
_axes_lead = [_fig.add_subplot(_gs[0, 1])]
_axes_lead += [_fig.add_subplot(_gs[i, 1], sharex=_axes_lead[0]) for i in (1, 2)]
plt.close(_fig)  # keep the object around without auto-displaying a stray copy

def redraw(*_):
    hr, gain, noise, cond_name = hr_slider.value, gain_slider.value, noise_slider.value, cond_dd.value
    cond = CONDITIONS[cond_name]
    amp_scale = float(gain)

    t_arr, leads, starts, rr_arr, window_ms = compute_leads(hr, gain, noise, cond)
    if time_slider.value > window_ms:      # clamp value BEFORE lowering max
        time_slider.value = 0.0
    time_slider.max = window_ms
    play.max = window_ms

    t_now = time_slider.value
    vec_now = vector_at_time(t_now, starts, rr_arr, cond, amp_scale)[0]

    ax_tri = _ax_tri
    ax_tri.clear()
    for ax in _axes_lead:
        ax.clear()

    # --- triangle + vector + projections (noise-free "true" cardiac vector) ---
    th = np.linspace(0, 2 * np.pi, 200)
    ax_tri.plot(np.cos(th), np.sin(th), color="lightgray", lw=1)  # spherical torso cross-section
    for name, pos in zip(("RA", "LA", "LL"), (RA, LA, LL)):
        ax_tri.plot(*pos, "ko", ms=6)
        ax_tri.annotate(name, pos, textcoords="offset points", xytext=(6, 6))

    proj = {}
    for lead, vec in LEAD_VECS.items():
        ax_tri.plot([-1.4 * vec[0], 1.4 * vec[0]], [-1.4 * vec[1], 1.4 * vec[1]],
                    "--", color="steelblue", lw=1, alpha=0.6)
        ax_tri.annotate(lead, 1.45 * vec, color="steelblue")
        proj[lead] = float(np.dot(vec_now, vec))
        p_pt = proj[lead] * vec
        ax_tri.plot([vec_now[0], p_pt[0]], [vec_now[1], p_pt[1]], ":", color="gray", lw=1)
        ax_tri.plot(*p_pt, "o", color="steelblue", ms=4)

    ax_tri.annotate("", xy=vec_now, xytext=(0, 0),
                     arrowprops=dict(arrowstyle="-|>", color="crimson", lw=2.5))
    ax_tri.set_xlim(-2, 2); ax_tri.set_ylim(-2, 2)
    ax_tri.set_aspect("equal"); ax_tri.axis("off")
    txt = "   ".join(f"{k}: {v:+.2f} mV" for k, v in proj.items())
    ax_tri.set_title(f"{cond_name}\n{txt}", fontsize=10)

    # --- lead waveforms (gain applied, noise added) ---
    for ax, key in zip(_axes_lead, ("I", "II", "III")):
        ax.plot(t_arr, leads[key], color="black", lw=1.2)
        ax.axvline(t_now, color="crimson", lw=1, alpha=0.8)
        ax.axhline(0, color="lightgray", lw=0.7)
        ax.set_ylabel(f"Lead {key}", fontsize=9)
        ax.set_ylim(-1.6 * max(1.0, gain), 1.6 * max(1.0, gain))
    _axes_lead[-1].set_xlabel("time (ms)")

    _fig.tight_layout()
    with out:
        out.clear_output(wait=True)
        display(_fig)

def _on_cond_change(change):
    hr_slider.value = CONDITIONS[cond_dd.value]["hr"]   # load a representative HR
    redraw()

def _jump(t_frac):
    def _cb(_b):
        time_slider.value = min(t_frac * (60000.0 / hr_slider.value), time_slider.max)
    return _cb

cond_dd.observe(_on_cond_change, names="value")
for w in (hr_slider, gain_slider, noise_slider, time_slider):
    w.observe(redraw, names="value")
btn_p.on_click(_jump(0.15))
btn_qrs.on_click(_jump(0.30))
btn_t.on_click(_jump(0.55))

ui = VBox([
    HBox([cond_dd, hr_slider, gain_slider, noise_slider]),
    HBox([play, time_slider, btn_p, btn_qrs, btn_t]),
    out,
])
display(ui)
redraw()

### Classic ECG-paper rhythm strip

Same underlying data as the Lead I/II/III panels above (via `compute_leads`), redrawn as a
single-lead strip on traditional ECG graph paper: pink minor gridlines every **40 ms / 0.1 mV**
(1 mm box) and bold major gridlines every **200 ms / 0.5 mV** (5 mm box) — the standard
25 mm/s, 10 mm/mV paper convention. P, Q, R, S, T peaks of the first beat are labeled
(P is omitted for Atrial Fibrillation, which has no discrete P wave). This model doesn't
include a U wave.

In [ ]:
PEAK_FRACS = {"P": 0.15, "Q": 0.28, "R": 0.30, "S": 0.33, "T": 0.55}  # fractions of RR, from vector_at_phase

def strip_peaks(cond, rr0):
    """Time (ms into the first beat) of each labeled wave; P is skipped when absent (AFib)."""
    return {name: frac * rr0 for name, frac in PEAK_FRACS.items() if name != "P" or cond["has_p"]}

def draw_ecg_grid(ax, t0, t1, y_span):
    """Classic pink ECG-paper grid: 1 mm boxes every 40 ms / 0.1 mV, 5 mm boxes every
    200 ms / 0.5 mV — the standard 25 mm/s, 10 mm/mV paper convention."""
    ax.set_facecolor("#fff5f8")
    for x in np.arange(0, t1 + 40, 40):
        ax.axvline(x, color="#f4b8c6", lw=0.4, zorder=0)
    for x in np.arange(0, t1 + 200, 200):
        ax.axvline(x, color="#e2607e", lw=0.8, zorder=0)
    for y in np.arange(-y_span, y_span + 0.1, 0.1):
        ax.axhline(y, color="#f4b8c6", lw=0.4, zorder=0)
    for y in np.arange(-y_span, y_span + 0.5, 0.5):
        ax.axhline(y, color="#e2607e", lw=0.8, zorder=0)

lead_dd = widgets.Dropdown(options=["I", "II", "III"], value="II", description="Strip lead")
out_strip = widgets.Output()

_fig_strip, _ax_strip = plt.subplots(figsize=(11, 3.2))
plt.close(_fig_strip)

def redraw_strip(*_):
    hr, gain, noise, cond_name = hr_slider.value, gain_slider.value, noise_slider.value, cond_dd.value
    cond = CONDITIONS[cond_name]
    lead = lead_dd.value

    t_arr, leads, starts, rr_arr, window_ms = compute_leads(hr, gain, noise, cond)
    wave = leads[lead]
    y_span = 1.6 * max(1.0, gain)

    ax = _ax_strip
    ax.clear()
    draw_ecg_grid(ax, t_arr[0], t_arr[-1], y_span)
    ax.plot(t_arr, wave, color="black", lw=1.4, zorder=3)

    t_now = min(time_slider.value, window_ms)
    ax.axvline(t_now, color="crimson", lw=1, alpha=0.8, zorder=2)

    peaks = strip_peaks(cond, rr_arr[0])
    peak_t = np.array(list(peaks.values()))
    peak_amp = vector_at_time(peak_t, starts, rr_arr, cond, float(gain)) @ LEAD_VECS[lead]
    for (name, t_ms), y in zip(peaks.items(), peak_amp):
        ax.plot(t_ms, y, "o", color="#1f6feb", ms=4, zorder=4)
        ax.annotate(name, (t_ms, y), textcoords="offset points", xytext=(0, 8 if y >= 0 else -14),
                    ha="center", fontsize=9, color="#1f6feb", weight="bold", zorder=5)

    ax.set_xlim(t_arr[0], t_arr[-1])
    ax.set_ylim(-y_span, y_span)
    ax.set_xlabel("time (ms)  —  25 mm/s paper-speed convention")
    ax.set_ylabel(f"Lead {lead} (mV-like units)")
    ax.set_title(f"{cond_name} — Lead {lead} rhythm strip", fontsize=10)

    _fig_strip.tight_layout()
    with out_strip:
        out_strip.clear_output(wait=True)
        display(_fig_strip)

for w in (hr_slider, gain_slider, noise_slider, cond_dd, lead_dd, time_slider):
    w.observe(redraw_strip, names="value")

display(VBox([lead_dd, out_strip]))
redraw_strip()